#### ⚙️ Step 1 — Install required libraries
Run the next cell once when you open the notebook for the first time.
You don't need to run it again unless you restart the runtime.

In [ ]:
# @title
!pip install requests pandas openpyxl -q
print("✅ Libraries installed")

#### ⚙️ Step 2 — Enter your EMA API credentials
Run the next cell.
Enter the credentials you received by email after registering on the EMA Account Management portal.
You will need: Token URL, Client ID, Client Secret, and Scope.
Click Connect — if successful, you will see a confirmation message. The token is valid for 1 hour.
If you get an error, double-check that you copied the credentials correctly.

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output

print("🔐 Enter your EMA API credentials")
print("─" * 45)

w_token_url    = widgets.Text(description="Token URL:", layout=widgets.Layout(width='700px'),
                              style={'description_width': '120px'})
w_client_id    = widgets.Text(description="Client ID:", layout=widgets.Layout(width='700px'),
                              style={'description_width': '120px'})
w_client_secret= widgets.Password(description="Client Secret:", layout=widgets.Layout(width='700px'),
                              style={'description_width': '120px'})
w_scope        = widgets.Text(description="Scope:", layout=widgets.Layout(width='700px'),
                              style={'description_width': '120px'})
btn_connect    = widgets.Button(description="Connect", button_style='primary')
out_connect    = widgets.Output()

display(w_token_url, w_client_id, w_client_secret, w_scope, btn_connect, out_connect)

import requests
token = None

def connect(b):
    global token
    with out_connect:
        clear_output()
        try:
            resp = requests.post(w_token_url.value.strip(), data={
                "grant_type":    "client_credentials",
                "client_id":     w_client_id.value.strip(),
                "client_secret": w_client_secret.value.strip(),
                "scope":         w_scope.value.strip()
            })
            resp.raise_for_status()
            token = resp.json()["access_token"]
            print("✅ Connected successfully! Token valid for 1 hour.")
        except Exception as e:
            print(f"❌ Error: {e}")

btn_connect.on_click(connect)

#### ⚙️ Step 3 — Search for medicinal products
Run the next cell. Fill in one or more fields and click Search.
All filters are combined with AND logic — the more fields you fill, the narrower the results.

Tips:
- Product name: free text (e.g. aspirin, ibuprofen, menarini)
- Substance, Country, Dose form: require numeric RMS/SMS codes (see reference files)
- ATC code: standard WHO ATC code (e.g. N02BA01)
- Status: filter by authorisation status

After searching, you will see a preview of the first results.
If you want full product details (MAH, country, dose form, substance strength, etc.),
click the "$everything" button — but read the warning first.

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import requests
import time
from urllib.parse import urlparse, parse_qs

BASE_URL = "https://api.pms.ema.europa.eu/public/v1"

STATUS_MAP = {
    "200000005005": "active",
    "200000005004": "withdrawn",
    "200000005006": "suspended",
    "200000005007": "refused",
    "200000005008": "expired"
}

STATUS_MAP_AUTH = {
    "100000072113": "valid",
    "100000072114": "withdrawn",
    "100000072115": "suspended",
    "100000072116": "refused",
    "100000072117": "expired"
}

df_results    = None
df_everything = None

print("🔍 EMA PMS Search")
print("─" * 55)
print("Fill in one or more fields. All filters are combined (AND logic).")
print()

style = {'description_width': '160px'}
lay   = widgets.Layout(width='550px')

w_name         = widgets.Text(description="Product name:",       placeholder="e.g. aspirin",                           layout=lay, style=style)
w_pms_id       = widgets.Text(description="PMS ID:",             placeholder="e.g. 600001772175",                      layout=lay, style=style)
w_substance    = widgets.Text(description="Substance (SMS ID):", placeholder="e.g. 100000090365 (ibuprofen)",          layout=lay, style=style)
w_country      = widgets.Text(description="Country (RMS ID):",   placeholder="e.g. 100000000388 (Spain)",              layout=lay, style=style)
w_dose_form    = widgets.Text(description="Dose form (RMS ID):", placeholder="e.g. 100000073665 (film-coated tablet)", layout=lay, style=style)
w_atc          = widgets.Text(description="ATC code:",           placeholder="e.g. N02BA01",                           layout=lay, style=style)
w_ma_number    = widgets.Text(description="MA number:",          placeholder="e.g. EU/1/00/001/001",                   layout=lay, style=style)
w_status       = widgets.Dropdown(description="Status:",         options=[("Any",""),("Active","200000005005"),("Withdrawn","200000005004"),("Suspended","200000005006"),("Refused","200000005007")], layout=lay, style=style)
w_last_updated = widgets.Text(description="Updated after:",      placeholder="e.g. 2024-01-01",                        layout=lay, style=style)
w_count        = widgets.Dropdown(description="Results per page:", options=[10,25,50,100], value=25,                   layout=lay, style=style)
w_sort         = widgets.Dropdown(description="Sort by:",        options=[("Name A-Z","name"),("Name Z-A","-name"),("Last updated","-lastUpdated"),("ID asc","id")], layout=lay, style=style)

btn_search     = widgets.Button(description="Search",   button_style='primary', icon='search', layout=widgets.Layout(width='140px', height='36px'))
btn_clear      = widgets.Button(description="Clear",    button_style='',        icon='times',  layout=widgets.Layout(width='120px', height='36px'))
out_search     = widgets.Output()

# $everything button — hidden until search returns results
btn_everything = widgets.Button(
    description="⚠️ Get full data ($everything)",
    button_style='warning',
    layout=widgets.Layout(width='280px', height='36px', display='none')
)
out_everything = widgets.Output()
lbl_everything = widgets.HTML(value="", layout=widgets.Layout(display='none'))

def get_params():
    params = {"_sort": w_sort.value}
    if w_name.value.strip():         params["name"]                 = w_name.value.strip()
    if w_pms_id.value.strip():       params["identifier"]           = w_pms_id.value.strip()
    if w_substance.value.strip():    params["_has:Ingredient:for:substance-code"] = w_substance.value.strip()
    if w_country.value.strip():      params["country"]              = w_country.value.strip()
    if w_dose_form.value.strip():    params["authorized-dose-form"] = w_dose_form.value.strip()
    if w_atc.value.strip():          params["_has:RegulatedAuthorization:subject:atc-classification"] = w_atc.value.strip()
    if w_ma_number.value.strip():    params["authorization-number"] = w_ma_number.value.strip()
    if w_status.value:               params["status"]               = w_status.value
    if w_last_updated.value.strip(): params["_lastUpdated"]         = f"gt{w_last_updated.value.strip()}"
    return params

def clear_form(b):
    global df_results, df_everything
    df_results = None
    df_everything = None
    for w in [w_name, w_pms_id, w_substance, w_country, w_dose_form, w_atc, w_ma_number, w_last_updated]:
        w.value = ''
    w_status.value = ''
    w_count.value  = 25
    btn_everything.layout.display = 'none'
    lbl_everything.layout.display = 'none'
    lbl_everything.value = ''
    with out_search:
        clear_output()
    with out_everything:
        clear_output()

btn_clear.on_click(clear_form)

def search(b):
    global df_results, df_everything
    df_everything = None
    with out_search:
        clear_output()
        if token is None:
            print("❌ Not connected. Run Cell 2 first.")
            return

        params = get_params()
        params["_count"] = w_count.value

        if len(params) <= 2:
            print("⚠️ Enter at least one search criterion.")
            return

        print("Searching...")
        try:
            resp = requests.get(
                f"{BASE_URL}/MedicinalProductDefinition",
                headers={"Authorization": f"Bearer {token}"},
                params=params
            )
            resp.raise_for_status()
            data    = resp.json()
            entries = data.get("entry", [])
            total   = data.get("total", len(entries))

            if not entries:
                print("No results found.")
                btn_everything.layout.display = 'none'
                lbl_everything.layout.display = 'none'
                return

            rows = []
            for e in entries:
                r    = e.get("resource", {})
                code = r.get("status", {}).get("coding", [{}])[0].get("code", "")
                rows.append({
                    "PMS ID": r.get("id", "-"),
                    "Name":   r.get("name", [{}])[0].get("productName", "-"),
                    "Status": STATUS_MAP.get(code, code)
                })
            df_results = pd.DataFrame(rows)
            print(f"✅ {total:,} total results — showing {len(df_results)} (use Cell 4 to download all)")
            display(df_results)

            # Show $everything button with warning if large
            btn_everything.layout.display = 'inline-flex'
            lbl_everything.layout.display = 'block'
            if total > 200:
                lbl_everything.value = (
                    f"<div style='margin-top:10px; padding:10px; background:#fff3cd; border-left:4px solid #ffc107; border-radius:4px; font-size:13px'>"
                    f"⚠️ <b>Warning:</b> {total:,} total results found. Using <b>$everything</b> requires 1 API request per product. "
                    f"With a limit of 500 req/min this could take <b>~{total//500 + 1} minute(s)</b> and may hit rate limits. "
                    f"Consider narrowing your search first."
                    f"</div>"
                )
            else:
                lbl_everything.value = (
                    f"<div style='margin-top:10px; padding:10px; background:#d1ecf1; border-left:4px solid #17a2b8; border-radius:4px; font-size:13px'>"
                    f"ℹ️ <b>$everything</b> will fetch full details for all {total:,} products "
                    f"(dose form, MAH, country, substance, strength, etc.). "
                    f"This will make {total:,} additional API requests."
                    f"</div>"
                )

        except Exception as e:
            err = ""
            try: err = resp.json().get("issue", [{}])[0].get("diagnostics", "")
            except: pass
            print(f"❌ Error: {err or e}")

btn_search.on_click(search)

def parse_everything(bundle):
    row = {}
    for entry in bundle.get("entry", []):
        r  = entry.get("resource", {})
        rt = r.get("resourceType", "")

        if rt == "MedicinalProductDefinition":
            row["PMS ID"]               = r.get("id", "")
            row["Name"]                 = r.get("name", [{}])[0].get("productName", "")
            sc                          = r.get("status", {}).get("coding", [{}])[0].get("code", "")
            row["Status"]               = STATUS_MAP.get(sc, sc)
            row["Dose form (code)"]     = r.get("combinedPharmaceuticalDoseForm", {}).get("coding", [{}])[0].get("code", "")
            row["Pediatric use"]        = r.get("pediatricUseIndicator", {}).get("coding", [{}])[0].get("code", "")
            row["Additional monitoring"]= r.get("additionalMonitoringIndicator", {}).get("coding", [{}])[0].get("code", "")
            row["Last updated"]         = r.get("meta", {}).get("lastUpdated", "")

        elif rt == "RegulatedAuthorization":
            subjects = r.get("subject", [{}])
            if any("MedicinalProductDefinition" in s.get("reference", "") for s in subjects):
                row["MA Number"]        = r.get("identifier", [{}])[0].get("value", "")
                row["Country (code)"]   = r.get("region", [{}])[0].get("coding", [{}])[0].get("code", "")
                ac                      = r.get("status", {}).get("coding", [{}])[0].get("code", "")
                row["Auth status"]      = STATUS_MAP_AUTH.get(ac, ac)
                row["Auth status date"] = r.get("statusDate", "")
                row["MAH"]              = r.get("holder", {}).get("display", "")
                row["Regulator"]        = r.get("regulator", {}).get("display", "")

        elif rt == "AdministrableProductDefinition":
            row["Route of admin (code)"] = r.get("routeOfAdministration", [{}])[0].get("code", {}).get("coding", [{}])[0].get("code", "")

        elif rt == "Ingredient":
            row["Substance (SMS ID)"]   = r.get("substance", {}).get("code", {}).get("concept", {}).get("coding", [{}])[0].get("code", "")
            strengths = r.get("substance", {}).get("strength", [{}])
            if strengths:
                s   = strengths[0]
                num = s.get("presentationRatio", {}).get("numerator", {})
                den = s.get("presentationRatio", {}).get("denominator", {})
                row["Strength value"]   = num.get("value", "")
                row["Strength unit"]    = num.get("code", "")
                row["Strength per"]     = den.get("code", "")
    return row

def run_everything(b):
    global df_everything
    with out_everything:
        clear_output()
        if token is None:
            print("❌ Not connected.")
            return

        params_base = get_params()
        params_base["_count"] = 100
        ct      = None
        page    = 1
        pms_ids = []

        print("Step 1: collecting all PMS IDs...")
        while True:
            p = params_base.copy()
            if ct:
                p["_ct"] = ct
            resp = requests.get(
                f"{BASE_URL}/MedicinalProductDefinition",
                headers={"Authorization": f"Bearer {token}"},
                params=p
            )
            resp.raise_for_status()
            data    = resp.json()
            entries = data.get("entry", [])
            for e in entries:
                pms_ids.append(e.get("resource", {}).get("id", ""))
            print(f"  Page {page} — {len(pms_ids)} IDs collected...")
            next_link = next((l for l in data.get("link", []) if l["relation"] == "next"), None)
            if not next_link:
                break
            ct = parse_qs(urlparse(next_link["url"]).query).get("_ct", [None])[0]
            if not ct:
                break
            page += 1

        print(f"\nStep 2: fetching full data for {len(pms_ids)} products...")
        all_rows = []
        for i, pms_id in enumerate(pms_ids):
            try:
                resp = requests.get(
                    f"{BASE_URL}/MedicinalProductDefinition/{pms_id}/$everything",
                    headers={"Authorization": f"Bearer {token}"}
                )
                resp.raise_for_status()
                all_rows.append(parse_everything(resp.json()))
                if (i + 1) % 25 == 0:
                    print(f"  {i+1}/{len(pms_ids)} done...")
                time.sleep(0.17)
            except Exception as e:
                all_rows.append({"PMS ID": pms_id, "Error": str(e)})

        df_everything = pd.DataFrame(all_rows)
        print(f"\n✅ Done! {len(df_everything)} products with full data.")
        print("👉 Now run Cell 4 to download the Excel.")
        display(df_everything.head(5))

btn_everything.on_click(run_everything)

display(
    w_name, w_pms_id, w_substance, w_country, w_dose_form,
    w_atc, w_ma_number, w_status, w_last_updated,
    w_count, w_sort,
    widgets.HBox([btn_search, btn_clear]),
    out_search,
    lbl_everything,
    btn_everything,
    out_everything
)

#### ⚠️ Downloading results

Next cell downloads all results with basic columns (PMS ID, Name, Status) using automatic pagination — 100 products per page.

#####📥 Want more data per product?

The API offers a second endpoint ($everything) that returns full details per product:

Dose form, route of administration
MA number, country, authorization status
MAH (marketing authorisation holder), regulator
Substance SMS ID and strength

However, this requires 1 additional API request per product. With a rate limit of 500 requests/minute, a search returning 400+ products may hit the limit and fail.
Recommendation: Use $everything only for small result sets (< 200 products). A separate notebook cell for this will be provided — use it intentionally.

#### ⚙️ Step 4 — Download results to Excel
Run this cell to download your results as an Excel file.

- If you used the $everything button in Step 3: downloads full data with all columns.
- If you didn't: downloads basic data (PMS ID, Name, Status) with automatic pagination — all pages, not just the preview.

The file will be saved to your Downloads folder automatically.

In [ ]:
# @title
from google.colab import files
from urllib.parse import urlparse, parse_qs

def download_all_basic(token, params_base):
    all_rows = []
    params = params_base.copy()
    params["_count"] = 100
    ct = None
    page = 1

    while True:
        if ct:
            params["_ct"] = ct
        elif "_ct" in params:
            del params["_ct"]

        resp = requests.get(
            f"{BASE_URL}/MedicinalProductDefinition",
            headers={"Authorization": f"Bearer {token}"},
            params=params
        )
        resp.raise_for_status()
        data    = resp.json()
        entries = data.get("entry", [])

        for e in entries:
            r    = e.get("resource", {})
            code = r.get("status", {}).get("coding", [{}])[0].get("code", "")
            all_rows.append({
                "PMS ID": r.get("id", "-"),
                "Name":   r.get("name", [{}])[0].get("productName", "-"),
                "Status": STATUS_MAP.get(code, code)
            })

        print(f"  Page {page} — {len(all_rows)} products so far...")
        next_link = next((l for l in data.get("link", []) if l["relation"] == "next"), None)
        if not next_link:
            break
        ct = parse_qs(urlparse(next_link["url"]).query).get("_ct", [None])[0]
        if not ct:
            break
        page += 1

    return pd.DataFrame(all_rows)


# ── Run ───────────────────────────────────────────────────────────────────────
if token is None:
    print("❌ Not connected. Run Cell 2 first.")
elif df_results is None:
    print("⚠️ Run a search first in Cell 3.")
else:
    params_base = get_params()

    if df_everything is not None and not df_everything.empty:
        # Already have full data from $everything
        filename = "ema_results_full.xlsx"
        df_everything.to_excel(filename, index=False)
        files.download(filename)
        print(f"✅ Downloaded full data: {filename} ({len(df_everything)} rows, {len(df_everything.columns)} columns)")
    else:
        # Download all pages with basic columns
        print("Downloading all pages (basic columns)...")
        df_all = download_all_basic(token, params_base)
        print(f"\n✅ Total: {len(df_all)} products")
        filename = "ema_results_basic.xlsx"
        df_all.to_excel(filename, index=False)
        files.download(filename)
        print(f"✅ Downloaded {filename}")